# Training Model LSTM untuk Prediksi Cuaca / Agroklimat Harian
**Tesis:** Model Hybrid LSTM - Random Forest untuk Pemilihan Varietas Tanaman Unggulan  
**Lokasi Studi:** Kabupaten Aceh Utara (5 Titik Pengamatan: Lhoksukon, Matangkuli, Syamtalira Aron, Dewantara, Kuta Makmur)  
**Sumber Data:** NASA POWER Agroclimatology (`data/raw/lstm/`)

---
### Ringkasan Alur Eksperimen:
1. **Memuat Data Agroklimat Harian Terpisah:**
   - **Data Training (2020 - 2024):** 9.135 baris data (5 lokasi x 1.827 hari)
   - **Data Testing (2025):** 1.825 baris data (5 lokasi x 365 hari)
2. **6 Parameter Cuaca (Multivariate Time-Series):**
   - Curah Hujan (`curah_hujan`), Suhu Rata-rata (`suhu_rata_rata`), Suhu Maksimal (`suhu_maksimal`), Suhu Minimal (`suhu_minimum`), Kelembaban Relatif (`kelembaban_rata_rata`), Radiasi Penyinaran Matahari (`lama_penyinaran`).
3. **Arsitektur Model LSTM (Sesuai Proposal Tesis):**
   - Input Window: 30 hari ke belakang ($t-29$ s/d $t$)
   - LSTM Layer 1: 128 hidden units + Dropout (0.2)
   - LSTM Layer 2: 64 hidden units + Dropout (0.2)
   - Dense Bottleneck: 32 unit (Vektor Representasi Temporal)
   - Output Layer: 6 output untuk memprediksi seluruh parameter agroklimat hari berikutnya ($t+1$)
4. **Evaluasi Model:** Metrik MAE, RMSE, $R^2$, MAPE, dan visualisasi Aktual vs Prediksi tahun 2025.
5. **Ekspor Embedding Temporal:** Menyimpan vektor 32-dimensi untuk tahap klasifikasi Random Forest (Hybrid Model).


In [ ]:
# 1. Setup Library & Konfigurasi Lingkungan
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set Reproducibility Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch Version : {torch.__version__}')
print(f'Device          : {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')


In [ ]:
# 2. Pemuatan Dataset Training (2020-2024) dan Testing (2025)
# Jalur fleksibel (mendukung Google Colab maupun Local Workspace)
TRAIN_CANDIDATES = [
    'data/raw/lstm/training/aceh_utara_agroklimat_harian.csv',
    'data/raw/lstm/train/training.csv',
    'data/training.csv',
    'D:/tesis-pak-salahuddin/projekskripsi/data/raw/lstm/training/aceh_utara_agroklimat_harian.csv',
    'training.csv'
]

TEST_CANDIDATES = [
    'data/raw/lstm/testing/aceh_utara_agroklimat_harian.csv',
    'data/raw/lstm/testing/testing.csv',
    'data/testing.csv',
    'D:/tesis-pak-salahuddin/projekskripsi/data/raw/lstm/testing/aceh_utara_agroklimat_harian.csv',
    'testing.csv'
]

def find_file(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

train_file = find_file(TRAIN_CANDIDATES)
test_file = find_file(TEST_CANDIDATES)

if train_file is None or test_file is None:
    try:
        from google.colab import files
        print('File data tidak ditemukan di folder lokal. Silakan upload file dataset...')
        uploaded = files.upload()
        for f in uploaded.keys():
            if 'train' in f.lower():
                train_file = f
            elif 'test' in f.lower():
                test_file = f
    except ImportError:
        raise FileNotFoundError('Pastikan dataset training dan testing ada di data/raw/lstm/')

df_train = pd.read_csv(train_file)
df_test = pd.read_csv(test_file)

print(f'Dataset Train dimuat dari : {train_file} -> {df_train.shape[0]} baris, {df_train.shape[1]} kolom')
print(f'Dataset Test dimuat dari  : {test_file}  -> {df_test.shape[0]} baris, {df_test.shape[1]} kolom')
print(f'Rentang Waktu Train       : {df_train["tanggal"].min()} s/d {df_train["tanggal"].max()} (Tahun: {df_train["tahun"].unique()})')
print(f'Rentang Waktu Test        : {df_test["tanggal"].min()} s/d {df_test["tanggal"].max()} (Tahun: {df_test["tahun"].unique()})')


In [ ]:
# 3. Parameter Agroklimat & Eksplorasi Statistik
FEATURES = [
    'curah_hujan',
    'suhu_rata_rata',
    'suhu_maksimal',
    'suhu_minimum',
    'kelembaban_rata_rata',
    'lama_penyinaran'
]

print('--- Statistik Ringkas Data Training (2020-2024) ---')
display(df_train[FEATURES].describe().T[['mean', 'std', 'min', '50%', 'max']])

# Plot tren parameter cuaca (sampel Lokasi L001 - Lhoksukon)
sample_loc = 'L001'
df_sample = df_train[df_train['id_lokasi'] == sample_loc].copy()
df_sample['tanggal_dt'] = pd.to_datetime(df_sample['tanggal'])
df_sample = df_sample.sort_values('tanggal_dt')

plt.figure(figsize=(15, 6))
plt.subplot(2, 1, 1)
plt.plot(df_sample['tanggal_dt'], df_sample['curah_hujan'], color='royalblue', label='Curah Hujan (mm/hari)')
plt.title(f'Pola Curah Hujan & Suhu Harian Lokasi {sample_loc} (2020-2024)')
plt.ylabel('Curah Hujan (mm)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(df_sample['tanggal_dt'], df_sample['suhu_rata_rata'], color='crimson', label='Suhu Rata-rata (°C)')
plt.plot(df_sample['tanggal_dt'], df_sample['suhu_maksimal'], color='darkorange', alpha=0.7, label='Suhu Maksimal (°C)')
plt.plot(df_sample['tanggal_dt'], df_sample['suhu_minimum'], color='teal', alpha=0.7, label='Suhu Minimal (°C)')
plt.ylabel('Suhu (°C)')
plt.xlabel('Tanggal')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 4. Preprocessing Lanjutan: Cleaning, Transformasi Log1p, Fitur Musiman & Sliding Window
# 4a. Cek Missing Values & Duplikat
print('Missing values Train:', df_train[FEATURES].isna().sum().sum())
print('Missing values Test :', df_test[FEATURES].isna().sum().sum())
df_train = df_train.sort_values(['id_lokasi', 'tanggal']).drop_duplicates(subset=['id_lokasi', 'tanggal']).reset_index(drop=True)
df_test = df_test.sort_values(['id_lokasi', 'tanggal']).drop_duplicates(subset=['id_lokasi', 'tanggal']).reset_index(drop=True)
df_train[FEATURES] = df_train[FEATURES].interpolate(method='linear').bfill().ffill()
df_test[FEATURES] = df_test[FEATURES].interpolate(method='linear').bfill().ffill()

# 4b. Feature Engineering: Tambah fitur siklik musiman (sin/cos bulan) agar LSTM paham musim hujan/kemarau
for dframe in [df_train, df_test]:
    dframe['bulan_sin'] = np.sin(2 * np.pi * dframe['bulan'] / 12.0)
    dframe['bulan_cos'] = np.cos(2 * np.pi * dframe['bulan'] / 12.0)

INPUT_FEATURES = FEATURES + ['bulan_sin', 'bulan_cos']  # 8 fitur input
TARGET_FEATURES = FEATURES                                # 6 parameter cuaca yang diprediksi

# 4c. Transformasi log1p khusus curah_hujan (karena distribusi sangat miring/skewed dengan banyak 0 dan spike >100mm)
df_train_proc = df_train.copy()
df_test_proc = df_test.copy()
df_train_proc['curah_hujan'] = np.log1p(df_train_proc['curah_hujan'].clip(lower=0))
df_test_proc['curah_hujan'] = np.log1p(df_test_proc['curah_hujan'].clip(lower=0))

# 4d. Scaling (MinMaxScaler HANYA di-fit pada data train 2020-2024 untuk mencegah data leakage)
input_scaler = MinMaxScaler(feature_range=(0, 1))
target_scaler = MinMaxScaler(feature_range=(0, 1))

df_train_scaled = df_train_proc.copy()
df_test_scaled = df_test_proc.copy()

df_train_scaled[INPUT_FEATURES] = input_scaler.fit_transform(df_train_proc[INPUT_FEATURES])
df_test_scaled[INPUT_FEATURES] = input_scaler.transform(df_test_proc[INPUT_FEATURES])

target_scaler.fit(df_train_proc[TARGET_FEATURES])

WINDOW_SIZE = 30  # Riwayat 30 hari ke belakang -> prediksi hari ke-31 (t+1)

def create_sequences_by_location(df_scaled, in_cols, out_cols, window_size=30):
    """Membuat sliding window per lokasi pengamatan agar urutan waktu konsisten."""
    X_list, y_list, loc_list, dates_list = [], [], [], []
    for loc_id, group in df_scaled.groupby('id_lokasi'):
        group_sorted = group.sort_values('tanggal').reset_index(drop=True)
        in_arr = group_sorted[in_cols].to_numpy(dtype=np.float32)
        out_arr = group_sorted[out_cols].to_numpy(dtype=np.float32)
        dates_arr = group_sorted['tanggal'].to_numpy()
        
        n_samples = len(group_sorted) - window_size
        for i in range(n_samples):
            X_list.append(in_arr[i : i + window_size])
            y_list.append(out_arr[i + window_size])
            loc_list.append(loc_id)
            dates_list.append(dates_arr[i + window_size])
            
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32), loc_list, dates_list

X_train, y_train, locs_train, dates_train = create_sequences_by_location(df_train_scaled, INPUT_FEATURES, TARGET_FEATURES, WINDOW_SIZE)
X_test, y_test, locs_test, dates_test = create_sequences_by_location(df_test_scaled, INPUT_FEATURES, TARGET_FEATURES, WINDOW_SIZE)

print(f'Dimensi X_train : {X_train.shape} (Sampel, Window={WINDOW_SIZE}, Input Fitur={len(INPUT_FEATURES)})')
print(f'Dimensi y_train : {y_train.shape} (Target 6 Parameter Cuaca)')
print(f'Dimensi X_test  : {X_test.shape}')
print(f'Dimensi y_test  : {y_test.shape}')

class WeatherDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 32
train_dataset = WeatherDataset(X_train, y_train)
test_dataset = WeatherDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print('Preprocessing & DataLoader siap!')


In [ ]:
# 5. Arsitektur Model LSTM Agroklimat (2-Layer LSTM + Embedding Layer)
class WeatherLSTM(nn.Module):
    def __init__(self, input_dim=6, hidden_1=128, hidden_2=64, embedding_dim=32, output_dim=6, dropout=0.2):
        super(WeatherLSTM, self).__init__()
        # Layer 1 LSTM: 128 hidden units
        self.lstm1 = nn.LSTM(input_size=input_dim, hidden_size=hidden_1, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        
        # Layer 2 LSTM: 64 hidden units
        self.lstm2 = nn.LSTM(input_size=hidden_1, hidden_size=hidden_2, batch_first=True)
        self.dropout2 = nn.Dropout(dropout)
        
        # Dense Bottleneck: 32 unit (Vektor Temporal Embedding untuk Hybrid RF)
        self.fc_embed = nn.Linear(hidden_2, embedding_dim)
        self.relu = nn.ReLU()
        
        # Regressor Output: 6 parameter agroklimat
        self.out_layer = nn.Linear(embedding_dim, output_dim)
        
    def extract_embedding(self, x):
        """Ekstraksi vektor embedding temporal 32-dimensi."""
        out, _ = self.lstm1(x)
        out = self.dropout1(out)
        out, _ = self.lstm2(out)
        last_timestep = out[:, -1, :]  # Ambil representasi waktu terakhir
        last_timestep = self.dropout2(last_timestep)
        embedding = self.relu(self.fc_embed(last_timestep))
        return embedding

    def forward(self, x):
        embedding = self.extract_embedding(x)
        prediction = self.out_layer(embedding)
        return prediction

model = WeatherLSTM(
    input_dim=len(INPUT_FEATURES),
    hidden_1=128,
    hidden_2=64,
    embedding_dim=32,
    output_dim=len(TARGET_FEATURES),
    dropout=0.2
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total Parameter Dilatih: {total_params:,}')


In [ ]:
# 6. Pelatihan Model (Training Loop) dengan Early Stopping & Tracking
import time
criterion = nn.HuberLoss(delta=1.0)  # Robust terhadap nilai ekstrim curah hujan
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

EPOCHS = 40
best_test_loss = float('inf')
train_losses = []
test_losses = []
epoch_records = []

os.makedirs('models', exist_ok=True)
BEST_MODEL_PATH = 'models/best_lstm_weather.pth'

print('Memulai proses pelatihan model LSTM...')
start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    ep_start = time.time()
    model.train()
    running_train_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item() * len(batch_x)
        
    epoch_train_loss = running_train_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)
    
    # Validasi pada data test 2025
    model.eval()
    running_test_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            preds = model(batch_x)
            t_loss = criterion(preds, batch_y)
            running_test_loss += t_loss.item() * len(batch_x)
            
    epoch_test_loss = running_test_loss / len(test_dataset)
    test_losses.append(epoch_test_loss)
    scheduler.step(epoch_test_loss)
    
    ep_dur = time.time() - ep_start
    is_best = epoch_test_loss < best_test_loss
    if is_best:
        best_test_loss = epoch_test_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        saved_mark = ' [Best Model Saved]'
    else:
        saved_mark = ''
        
    epoch_records.append({
        'epoch': epoch,
        'train_loss': epoch_train_loss,
        'test_loss': epoch_test_loss,
        'duration_sec': ep_dur,
        'is_best': is_best
    })
    
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f'Epoch [{epoch:02d}/{EPOCHS}] | Train Loss: {epoch_train_loss:.5f} | Test Loss: {epoch_test_loss:.5f}{saved_mark}')

total_duration = time.time() - start_time
print(f'Pelatihan selesai dalam {total_duration:.2f} detik! Model terbaik disimpan di: {BEST_MODEL_PATH}')


In [ ]:
# 7. Evaluasi Model pada Data Testing 2025 (Termasuk Akurasi Persentase %)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

with torch.no_grad():
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_pred_scaled = model(X_test_tensor).cpu().numpy()

# Denormalisasi ke skala sebelum scaling
y_pred_real = target_scaler.inverse_transform(y_pred_scaled)
y_test_real = target_scaler.inverse_transform(y_test)

# Kembalikan log1p khusus kolom curah_hujan (indeks 0) ke mm/hari asli
y_pred_real[:, 0] = np.expm1(np.clip(y_pred_real[:, 0], a_min=0, a_max=None))
y_test_real[:, 0] = np.expm1(np.clip(y_test_real[:, 0], a_min=0, a_max=None))

# Ambang batas toleransi realistis per parameter untuk metrik Tolerance Accuracy (%)
TOLERANCES = {
    'curah_hujan': 5.0,           # selisih <= 5 mm/hari
    'suhu_rata_rata': 1.0,        # selisih <= 1.0 °C
    'suhu_maksimal': 1.5,         # selisih <= 1.5 °C
    'suhu_minimum': 1.5,          # selisih <= 1.5 °C
    'kelembaban_rata_rata': 5.0,  # selisih <= 5.0 %
    'lama_penyinaran': 3.0        # selisih <= 3.0 MJ/m2
}

metrics_list = []
for idx, col in enumerate(TARGET_FEATURES):
    actual = y_test_real[:, idx]
    pred = y_pred_real[:, idx]
    
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2 = r2_score(actual, pred)
    
    # 1. Akurasi berbasis WAPE (Weighted Absolute Percentage Error) - aman terhadap nilai 0
    wape = (np.sum(np.abs(actual - pred)) / (np.sum(np.abs(actual)) + 1e-8)) * 100
    acc_wape = max(0.0, 100.0 - wape)
    
    # 2. Akurasi berbasis sMAPE (Symmetric MAPE)
    denom = (np.abs(actual) + np.abs(pred)) / 2.0 + 1e-8
    smape = np.mean(np.abs(actual - pred) / denom) * 100
    acc_smape = max(0.0, 100.0 - smape)
    
    # 3. Tolerance Accuracy (% hari dengan error <= batas toleransi)
    tol = TOLERANCES.get(col, 1.0)
    acc_tol = np.mean(np.abs(actual - pred) <= tol) * 100
    
    metrics_list.append({
        'Parameter Cuaca': col,
        'MAE': round(mae, 3),
        'RMSE': round(rmse, 3),
        'R2 Score': round(r2, 3),
        'Akurasi WAPE (%)': round(acc_wape, 2),
        'Akurasi Toleransi (%)': round(acc_tol, 2),
        'Batas Toleransi': f'±{tol}'
    })

df_metrics = pd.DataFrame(metrics_list)
overall_wape_acc = df_metrics['Akurasi WAPE (%)'].mean()
overall_tol_acc = df_metrics['Akurasi Toleransi (%)'].mean()

# Akurasi Klasifikasi Kejadian Hujan vs Tidak Hujan (Threshold >= 0.5 mm)
rain_actual_cls = (y_test_real[:, 0] >= 0.5).astype(int)
rain_pred_cls = (y_pred_real[:, 0] >= 0.5).astype(int)
rain_cls_acc = np.mean(rain_actual_cls == rain_pred_cls) * 100

print('===================================================================================')
print('        METRIK EVALUASI & AKURASI PERSENTASE MODEL LSTM (TESTING 2025)             ')
print('===================================================================================')
display(df_metrics)
print(f'\n>> Rata-rata Akurasi Prediksi (100% - WAPE) : {overall_wape_acc:.2f}%')
print(f'>> Rata-rata Akurasi Toleransi             : {overall_tol_acc:.2f}%')
print(f'>> Akurasi Klasifikasi Hujan vs Cerah       : {rain_cls_acc:.2f}%')


In [ ]:
# 8. Visualisasi Hasil Prediksi vs Aktual Tahun 2025
os.makedirs('results', exist_ok=True)
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train Loss (2020-2024)', color='dodgerblue')
plt.plot(test_losses, label='Test Loss (2025)', color='darkorange', linestyle='--')
plt.title('Kurva Loss Pelatihan LSTM (Huber Loss)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('lstm_loss_curve.png', dpi=150)
plt.show()

# Visualisasi Plot Aktual vs Prediksi (120 hari pertama tahun 2025)
N_DAYS = 120
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

# 1. Curah Hujan
axes[0].plot(dates_test[:N_DAYS], y_test_real[:N_DAYS, 0], color='blue', alpha=0.8, label='Aktual Curah Hujan')
axes[0].plot(dates_test[:N_DAYS], y_pred_real[:N_DAYS, 0], color='red', linestyle='--', label='Prediksi LSTM')
axes[0].set_ylabel('Curah Hujan (mm)')
axes[0].set_title('Perbandingan Prediksi vs Aktual Agroklimat Harian 2025 (Sampel 120 Hari)')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# 2. Suhu Rata-rata
axes[1].plot(dates_test[:N_DAYS], y_test_real[:N_DAYS, 1], color='green', alpha=0.8, label='Aktual Suhu Rata-rata')
axes[1].plot(dates_test[:N_DAYS], y_pred_real[:N_DAYS, 1], color='darkorange', linestyle='--', label='Prediksi LSTM')
axes[1].set_ylabel('Suhu (°C)')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# 3. Kelembaban Rata-rata
axes[2].plot(dates_test[:N_DAYS], y_test_real[:N_DAYS, 4], color='purple', alpha=0.8, label='Aktual Kelembaban')
axes[2].plot(dates_test[:N_DAYS], y_pred_real[:N_DAYS, 4], color='coral', linestyle='--', label='Prediksi LSTM')
axes[2].set_ylabel('Kelembaban (%)')
axes[2].set_xlabel('Tanggal')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('lstm_weather_predictions_sample.png', dpi=150)
plt.show()


In [ ]:
# 9. Ekstraksi Vektor Embedding Temporal (32-Dimensi) untuk Model Hybrid RF
print('Mengekstraksi representasi fitur temporal 32-dimensi dari LSTM...')
model.eval()
with torch.no_grad():
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    
    emb_train = model.extract_embedding(X_train_tensor).cpu().numpy()
    emb_test = model.extract_embedding(X_test_tensor).cpu().numpy()

print(f'Dimensi Embedding Train : {emb_train.shape}')
print(f'Dimensi Embedding Test  : {emb_test.shape}')

# Simpan bundle embedding agar dapat langsung dimuat di notebook Random Forest
os.makedirs('data/processed', exist_ok=True)
EXPORT_PATH = 'data/processed/lstm_weather_embeddings.npz'
np.savez_compressed(
    EXPORT_PATH,
    emb_train=emb_train,
    emb_test=emb_test,
    dates_train=dates_train,
    dates_test=dates_test,
    locs_train=locs_train,
    locs_test=locs_test,
    features=FEATURES
)

print(f'Vektor embedding temporal sukses disimpan ke: {EXPORT_PATH}')
print('File ini siap digunakan oleh notebooks/training_random_forest_colab.ipynb untuk integrasi Hybrid!')

# Opsi Download file ke komputer lokal jika dijalankan di Google Colab:
# try:
#     from google.colab import files
#     files.download(BEST_MODEL_PATH)
#     files.download(EXPORT_PATH)
# except ImportError:
#     pass


In [ ]:
# 10. Pencatatan Log Eksperimen Otomatis (Standar Proyek & Google Colab)
import json
import csv
from datetime import datetime

run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Ringkasan Metrik Rata-rata
avg_metrics = {
    'mean_mae': round(float(df_metrics['MAE'].mean()), 3),
    'mean_rmse': round(float(df_metrics['RMSE'].mean()), 3),
    'mean_r2': round(float(df_metrics['R2 Score'].mean()), 3),
    'mean_accuracy_wape_pct': round(float(overall_wape_acc), 2),
    'mean_accuracy_tol_pct': round(float(overall_tol_acc), 2),
    'rain_classification_acc_pct': round(float(rain_cls_acc), 2),
    'detail_per_fitur': df_metrics.to_dict(orient='records')
}

hyperparams = {
    'hidden_1': 128,
    'hidden_2': 64,
    'embedding_dim': 32,
    'dropout': 0.2,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.001,
    'window_size': WINDOW_SIZE,
    'features': INPUT_FEATURES
}

# Simpan Riwayat Loss per Epoch
pd.DataFrame(epoch_records).to_csv('lstm_weather_loss_history.csv', index=False)
df_metrics.to_csv('lstm_weather_metrics.csv', index=False)

# 1. Format Baris Log Standar Proyek (results/experiment_history.csv)
log_row = {
    'run_id': run_id,
    'timestamp': timestamp,
    'git_commit': 'colab',
    'git_dirty': False,
    'stage': 'LSTM_Weather',
    'environment': 'google_colab' if 'google.colab' in str(get_ipython()) else 'local',
    'train_samples': len(X_train),
    'test_samples': len(X_test),
    'num_classes': len(FEATURES),  # 6 parameter agroklimat
    'hyperparameters': json.dumps(hyperparams, ensure_ascii=False),
    'loss_akhir': round(float(best_test_loss), 5),
    'accuracy': round(float(overall_wape_acc), 2),
    'precision_weighted': f'R2={avg_metrics["mean_r2"]}',
    'recall_weighted': f'MAE={avg_metrics["mean_mae"]}',
    'f1_weighted': f'RMSE={avg_metrics["mean_rmse"]}',
    'top_3_accuracy': f'AccWAPE={avg_metrics["mean_accuracy_wape_pct"]}%',
    'top_features': '; '.join(FEATURES),
    'duration_sec': round(float(total_duration), 2),
    'catatan': f'LSTM Prediksi Cuaca Harian Aceh Utara (window={WINDOW_SIZE} hari)'
}

# Simpan Colab Export Log CSV & JSON
pd.DataFrame([log_row]).to_csv('colab_experiment_history.csv', index=False)

summary_data = {
    'run_id': run_id,
    'timestamp': timestamp,
    'stage': 'LSTM_Weather',
    'duration_sec': round(float(total_duration), 2),
    'best_test_loss': float(best_test_loss),
    'hyperparameters': hyperparams,
    'metrics': avg_metrics
}
with open('colab_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=2, ensure_ascii=False)

# Jika running di lokal dengan repositori yang memiliki folder results/:
if os.path.exists('results'):
    local_history = 'results/experiment_history.csv'
    write_hdr = not os.path.exists(local_history)
    with open(local_history, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(log_row.keys()))
        if write_hdr:
            writer.writeheader()
        writer.writerow(log_row)
    
    run_dir = os.path.join('results', 'runs', f'{run_id}_LSTM_Weather')
    os.makedirs(run_dir, exist_ok=True)
    with open(os.path.join(run_dir, 'summary.json'), 'w', encoding='utf-8') as f:
        json.dump(summary_data, f, indent=2, ensure_ascii=False)
    print(f'[LOG LOKAL] Tersimpan di: {local_history} & {run_dir}')

print('========================================================================')
print('  LOGGING SELESAI: Riwayat & Metrik Tercatat Secara Otomatis!           ')
print('========================================================================')
print(f'Run ID          : {run_id}')
print(f'Durasi Training : {total_duration:.2f} detik')
print(f'Best Test Loss  : {best_test_loss:.5f}')
print(f'R2 Rata-rata    : {avg_metrics["mean_r2"]}')
print(f'MAE Rata-rata   : {avg_metrics["mean_mae"]}')
print(f'RMSE Rata-rata  : {avg_metrics["mean_rmse"]}')
print('File log: colab_experiment_history.csv, colab_summary.json, lstm_weather_loss_history.csv')


In [ ]:
# 11. Download Hasil Training & Log dari Google Colab ke Komputer Lokal
# Jalankan sel ini di Google Colab untuk mendownload model, embedding, dan log eksperimen
try:
    from google.colab import files
    print('Menyiapkan file untuk didownload...')
    files.download('models/best_lstm_weather.pth')
    files.download('data/processed/lstm_weather_embeddings.npz')
    files.download('colab_experiment_history.csv')
    files.download('colab_summary.json')
    files.download('lstm_weather_loss_history.csv')
    files.download('lstm_weather_metrics.csv')
    files.download('lstm_loss_curve.png')
    files.download('lstm_weather_predictions_sample.png')
    print('Semua file log, model, dan grafik berhasil dipicu untuk didownload!')
except ImportError:
    print('Sel ini khusus untuk Google Colab (files.download). Di lokal, semua file sudah tersimpan di direktori.')
